In [2]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from scipy.interpolate import CubicSpline

m, w, lam, QA, QB, T = 1.0, 1.0, 1.0, 1.0, -0.5, 2.0
dV  = lambda q: w**2 * q + lam * q**3
rhs = lambda t, y: [y[1], -dV(y[0]) / m]

def miss(v0):
    s = solve_ivp(rhs, (0, T), [QA, v0], rtol=1e-13, atol=1e-15)
    return s.y[0, -1] - QB

v0_star = brentq(miss, -4.0, -1.0, xtol=1e-15, rtol=8.9e-16)   # fundamental branch
sol     = solve_ivp(rhs, (0, T), [QA, v0_star], rtol=1e-13, atol=1e-15,
                    dense_output=True)
t_ref   = np.linspace(0, T, 20001)
q_true  = CubicSpline(t_ref, sol.sol(t_ref)[0])
print(f"v0* = {v0_star:.15f}")

v0* = -2.306319364952174


In [3]:
# one degree of freedom, anharmonic well:  L = ½ q̇² − V(q)
V  = lambda q: 0.5*q**2 + 0.25*q**4
dV = lambda q: q + q**3
N = 200

def action(q, h):
    qd = np.diff(q)/h                       # velocities on cell centers
    Vbar = 0.5*(V(q[:-1]) + V(q[1:]))       # trapezoid for the potential
    return np.sum(0.5*qd**2 - Vbar)*h

t  = np.linspace(0.0, T, N + 1)
h = t[1] - t[0]
q_guess = np.linspace(QA, QB, N + 1)          # not np.linspace(1.0, -5.0, ...)

assert np.isclose(q_guess[0],  QA) and np.isclose(q_guess[-1], QB), \
    "initial guess endpoints do not match the boundary conditions"
S0 = action(q_guess, h)
print("S0[straight line =", S0)

S0[straight line = 0.24373359380273435


In [4]:
shape_smooth = np.sin(np.pi*t/T)             # well-behaved mode

rng = np.random.default_rng(0)               # deliberately ugly
shape_random = rng.normal(size=t.size)
shape_random[0] = shape_random[-1] = 0.0     # MUST vanish at the endpoints
shape_random /= np.abs(shape_random).max()

for shape in (shape_smooth, shape_random):
    for eps in (0.2, 0.1, 0.05, 0.025):
        dS = action(q_true(t)+ eps*shape, h) - S0
        print(f"eps={eps:<7} dS={dS: .6e}  dS/eps={dS/eps: .4f}")

eps=0.2     dS= 1.473579e+00  dS/eps= 7.3679
eps=0.1     dS= 1.518152e+00  dS/eps= 15.1815
eps=0.05    dS= 1.530144e+00  dS/eps= 30.6029
eps=0.025   dS= 1.533251e+00  dS/eps= 61.3300
eps=0.2     dS= 1.249715e+02  dS/eps= 624.8577
eps=0.1     dS= 3.239366e+01  dS/eps= 323.9366
eps=0.05    dS= 9.249148e+00  dS/eps= 184.9830
eps=0.025   dS= 3.463017e+00  dS/eps= 138.5207


In [5]:
# q_true: the exact path with q(0)=0, q(1)=0.5  (provided in the notebook)
S0 = action(q_true(t), h)
print("S0[exact path =", S0)
shape_smooth = np.sin(np.pi*t/T)             # well-behaved mode

rng = np.random.default_rng(0)               # deliberately ugly
shape_random = rng.normal(size=t.size)
shape_random[0] = shape_random[-1] = 0.0     # MUST vanish at the endpoints
shape_random /= np.abs(shape_random).max()

for shape in (shape_smooth, shape_random):
    for eps in (0.2, 0.1, 0.05, 0.025):
        dS = action(q_true(t)+ eps*shape, h) - S0
        print(f"eps={eps:<7} dS={dS: .6e}  dS/eps={dS/eps: .4f} dS/eps^2={dS/eps**2: .4f}")

S0[exact path = 1.7780396829263412
eps=0.2     dS=-6.072753e-02  dS/eps=-0.3036 dS/eps^2=-1.5182
eps=0.1     dS=-1.615395e-02  dS/eps=-0.1615 dS/eps^2=-1.6154
eps=0.05    dS=-4.161601e-03  dS/eps=-0.0832 dS/eps^2=-1.6646
eps=0.025   dS=-1.055053e-03  dS/eps=-0.0422 dS/eps^2=-1.6881
eps=0.2     dS= 1.234372e+02  dS/eps= 617.1862 dS/eps^2= 3085.9310
eps=0.1     dS= 3.085935e+01  dS/eps= 308.5935 dS/eps^2= 3085.9351
eps=0.05    dS= 7.714842e+00  dS/eps= 154.2968 dS/eps^2= 3085.9369
eps=0.025   dS= 1.928711e+00  dS/eps= 77.1484 dS/eps^2= 3085.9376


In [6]:
Vh = lambda q: 0.5*q**2
def action_h(q, h):
    qd = np.diff(q)/h
    Vbar = 0.5*(Vh(q[:-1]) + Vh(q[1:]))
    return np.sum(0.5*qd**2 - Vbar)*h

eps = 1e-3
for T in (1.0, 2.0, 3.0, np.pi, 3.5, 4.0, 5.0):
    t = np.linspace(0, T, 4001); h = t[1]-t[0]
    q = eps*np.sin(np.pi*t/T)                # true path is q ≡ 0
    print(f"T={T:<8.5f}  S/eps^2 = {action_h(q,h)/eps**2: .6f}")

T=1.00000   S/eps^2 =  2.217401
T=2.00000   S/eps^2 =  0.733700
T=3.00000   S/eps^2 =  0.072467
T=3.14159   S/eps^2 = -0.000000
T=3.50000   S/eps^2 = -0.170028
T=4.00000   S/eps^2 = -0.383150
T=5.00000   S/eps^2 = -0.756520
